# Homework 3: Methods for Classification
# Air Quality Level Classification using Machine Learning

**IEEE Conference Paper Format**

This notebook follows the IEEE paper structure:
- Abstract
- Introduction
- Methods
- Results

## Abstract

*To be completed: Provide a short and informative overview of the classification methods applied, the problem being solved, and key performance results obtained.*

## 1. Introduction

*To be completed:*
- *Importance of air quality level classification*
- *Brief literature review on classification methods for environmental data*
- *Problem definition: Define the classes (e.g., Good, Moderate, Unhealthy air quality)*
- *Motivation and applications*
- *References to related work*

## 2. Methods

This section describes the classification methods and evaluation techniques used in this study.

### 2.1 Setup and Imports

In [ ]:
# Standard libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Classification models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

# Preprocessing and model selection
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve,
    precision_recall_curve, auc
)

# Custom database module
import sys
sys.path.append('..')
from database.data_loader import DataLoader

# Display settings
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
%matplotlib inline

### 2.2 Data Loading and Preparation

In [ ]:
# Initialize data loader
loader = DataLoader(data_path='../data')

# Load and prepare Air Quality dataset
# df = loader.load_csv('AirQualityUCI.csv', sep=';', decimal=',')

# For demonstration, create sample data
np.random.seed(42)
n_samples = 9358
df = pd.DataFrame({
    'CO(GT)': np.random.uniform(0.5, 5.0, n_samples),
    'PT08.S1(CO)': np.random.uniform(800, 1600, n_samples),
    'C6H6(GT)': np.random.uniform(2, 20, n_samples),
    'PT08.S2(NMHC)': np.random.uniform(600, 1400, n_samples),
    'NOx(GT)': np.random.uniform(50, 300, n_samples),
    'PT08.S3(NOx)': np.random.uniform(400, 1200, n_samples),
    'NO2(GT)': np.random.uniform(20, 150, n_samples),
    'PT08.S4(NO2)': np.random.uniform(800, 1800, n_samples),
    'PT08.S5(O3)': np.random.uniform(600, 1600, n_samples),
    'T': np.random.uniform(5, 35, n_samples),
    'RH': np.random.uniform(20, 80, n_samples),
    'AH': np.random.uniform(0.3, 1.8, n_samples)
})

print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
df.head()

### 2.3 Target Variable Creation

Create classification labels based on air quality thresholds.

In [ ]:
# Create air quality categories based on CO levels (example)
# Define thresholds
def classify_air_quality(co_value):
    if co_value < 2.0:
        return 'Good'
    elif co_value < 3.5:
        return 'Moderate'
    else:
        return 'Unhealthy'

df['AirQuality'] = df['CO(GT)'].apply(classify_air_quality)

# Display class distribution
print("Air Quality Class Distribution:")
print(df['AirQuality'].value_counts())
print(f"\nClass proportions:")
print(df['AirQuality'].value_counts(normalize=True))

# Visualize class distribution
plt.figure(figsize=(8, 5))
df['AirQuality'].value_counts().plot(kind='bar', color=['green', 'orange', 'red'])
plt.title('Air Quality Class Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Air Quality Level', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### 2.4 Feature and Target Preparation

In [ ]:
# Define features (exclude CO since it was used to create the target)
feature_cols = [col for col in df.columns if col not in ['CO(GT)', 'AirQuality']]

X = df[feature_cols]
y = df['AirQuality']

print(f"Features: {feature_cols}")
print(f"\nFeature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")

### 2.5 Train-Test Split

In [ ]:
# Split data with stratification to maintain class proportions
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set: {X_test.shape[0]} samples")
print(f"\nTraining class distribution:")
print(y_train.value_counts())
print(f"\nTesting class distribution:")
print(y_test.value_counts())

### 2.6 Feature Scaling

In [ ]:
# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed")

### 2.7 Classification Models

We will implement and compare multiple classification algorithms.

#### 2.7.1 Logistic Regression

In [ ]:
# Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)

# Predictions
y_pred_lr = lr_model.predict(X_test_scaled)

# Evaluation
lr_accuracy = accuracy_score(y_test, y_pred_lr)
lr_precision = precision_score(y_test, y_pred_lr, average='weighted')
lr_recall = recall_score(y_test, y_pred_lr, average='weighted')
lr_f1 = f1_score(y_test, y_pred_lr, average='weighted')

print("Logistic Regression Results:")
print(f"Accuracy: {lr_accuracy:.4f}")
print(f"Precision: {lr_precision:.4f}")
print(f"Recall: {lr_recall:.4f}")
print(f"F1-Score: {lr_f1:.4f}")

#### 2.7.2 Decision Tree

In [ ]:
# Decision Tree
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)

# Predictions
y_pred_dt = dt_model.predict(X_test)

# Evaluation
dt_accuracy = accuracy_score(y_test, y_pred_dt)
dt_precision = precision_score(y_test, y_pred_dt, average='weighted')
dt_recall = recall_score(y_test, y_pred_dt, average='weighted')
dt_f1 = f1_score(y_test, y_pred_dt, average='weighted')

print("Decision Tree Results:")
print(f"Accuracy: {dt_accuracy:.4f}")
print(f"Precision: {dt_precision:.4f}")
print(f"Recall: {dt_recall:.4f}")
print(f"F1-Score: {dt_f1:.4f}")

#### 2.7.3 Random Forest

In [ ]:
# Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Predictions
y_pred_rf = rf_model.predict(X_test)

# Evaluation
rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_precision = precision_score(y_test, y_pred_rf, average='weighted')
rf_recall = recall_score(y_test, y_pred_rf, average='weighted')
rf_f1 = f1_score(y_test, y_pred_rf, average='weighted')

print("Random Forest Results:")
print(f"Accuracy: {rf_accuracy:.4f}")
print(f"Precision: {rf_precision:.4f}")
print(f"Recall: {rf_recall:.4f}")
print(f"F1-Score: {rf_f1:.4f}")

#### 2.7.4 Support Vector Machine (SVM)

In [ ]:
# SVM
svm_model = SVC(kernel='rbf', random_state=42)
svm_model.fit(X_train_scaled, y_train)

# Predictions
y_pred_svm = svm_model.predict(X_test_scaled)

# Evaluation
svm_accuracy = accuracy_score(y_test, y_pred_svm)
svm_precision = precision_score(y_test, y_pred_svm, average='weighted')
svm_recall = recall_score(y_test, y_pred_svm, average='weighted')
svm_f1 = f1_score(y_test, y_pred_svm, average='weighted')

print("SVM Results:")
print(f"Accuracy: {svm_accuracy:.4f}")
print(f"Precision: {svm_precision:.4f}")
print(f"Recall: {svm_recall:.4f}")
print(f"F1-Score: {svm_f1:.4f}")

#### 2.7.5 Gradient Boosting

In [ ]:
# Gradient Boosting
gb_model = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb_model.fit(X_train, y_train)

# Predictions
y_pred_gb = gb_model.predict(X_test)

# Evaluation
gb_accuracy = accuracy_score(y_test, y_pred_gb)
gb_precision = precision_score(y_test, y_pred_gb, average='weighted')
gb_recall = recall_score(y_test, y_pred_gb, average='weighted')
gb_f1 = f1_score(y_test, y_pred_gb, average='weighted')

print("Gradient Boosting Results:")
print(f"Accuracy: {gb_accuracy:.4f}")
print(f"Precision: {gb_precision:.4f}")
print(f"Recall: {gb_recall:.4f}")
print(f"F1-Score: {gb_f1:.4f}")

### 2.8 Model Comparison

In [ ]:
# Compare all models
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree', 'Random Forest', 'SVM', 'Gradient Boosting'],
    'Accuracy': [lr_accuracy, dt_accuracy, rf_accuracy, svm_accuracy, gb_accuracy],
    'Precision': [lr_precision, dt_precision, rf_precision, svm_precision, gb_precision],
    'Recall': [lr_recall, dt_recall, rf_recall, svm_recall, gb_recall],
    'F1-Score': [lr_f1, dt_f1, rf_f1, svm_f1, gb_f1]
})

results = results.sort_values('F1-Score', ascending=False)
print("\nModel Performance Comparison:")
print("=" * 90)
print(results.to_string(index=False))

# Visualize comparison
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(results))
width = 0.2

ax.bar(x - 1.5*width, results['Accuracy'], width, label='Accuracy')
ax.bar(x - 0.5*width, results['Precision'], width, label='Precision')
ax.bar(x + 0.5*width, results['Recall'], width, label='Recall')
ax.bar(x + 1.5*width, results['F1-Score'], width, label='F1-Score')

ax.set_xlabel('Models', fontsize=12)
ax.set_ylabel('Scores', fontsize=12)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(results['Model'], rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### 2.9 Confusion Matrix Analysis

In [ ]:
# Confusion matrices for all models
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.ravel()

models_preds = [
    ('Logistic Regression', y_pred_lr),
    ('Decision Tree', y_pred_dt),
    ('Random Forest', y_pred_rf),
    ('SVM', y_pred_svm),
    ('Gradient Boosting', y_pred_gb)
]

for idx, (name, y_pred) in enumerate(models_preds):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx])
    axes[idx].set_title(f'{name}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

# Remove extra subplot
fig.delaxes(axes[5])

plt.tight_layout()
plt.show()

### 2.10 Feature Importance

In [ ]:
# Feature importance from Random Forest
feature_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

# Plot feature importance
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['Feature'], feature_importance['Importance'])
plt.xlabel('Importance', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.title('Feature Importance from Random Forest', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\nTop 5 Important Features:")
print(feature_importance.head())

### 2.11 Classification Report

In [ ]:
# Detailed classification report for best model
print("Classification Report for Random Forest:")
print("=" * 80)
print(classification_report(y_test, y_pred_rf))

## 3. Results

*To be completed: Discuss your findings, including:*
- *Comparison of classification models across all metrics*
- *Best performing model and why*
- *Confusion matrix interpretation for each class*
- *Feature importance insights*
- *Model strengths and limitations*
- *Practical implications for air quality monitoring*

## 4. Conclusions

*To be completed: Summarize the main findings, best model selection, and recommendations for air quality classification.*

## 5. References

*To be completed: Add references to:*
- *UCI Machine Learning Repository*
- *Relevant papers on classification methods*
- *Air quality classification studies*
- *Algorithm documentation*